## Task 1
Initialize the Feast Feature Repository

Set up a Feast feature repository within your existing IRIS pipeline project. Configure it to use a local backend (SQLite is sufficient). Ensure the repository structure follows Feast conventions and is committed to Git.

In [ ]:
feast init feature_repo

## Task 2
Define Entities, Data Sources & Feature Views

Within your feature repository, define:

An entity that uniquely identifies each iris sample. A data source pointing to your IRIS dataset. A feature view that maps the iris feature columns to the entity and data source.
Think carefully about what timestamp field Feast requires and how you will provide one for a static dataset like IRIS.

In [ ]:
pwd

In [ ]:
#Read and print into to complete feature view and since feast can only handle parquet file, we need to update it. 
import pandas as pd

data=pd.read_csv("feature_repo/data/iris_data_adapted_for_feast.csv")

In [ ]:
data.info()

In [ ]:
#since event timestamp becomes an issue, we need to convert it to the datetime
data['event_timestamp'] = pd.to_datetime(data['event_timestamp'],utc=True)
data['created_timestamp'] = pd.to_datetime(data['created_timestamp'],utc=True)

In [ ]:
data.to_parquet("feature_repo/data/iris.parquet", index=False)

## Task 3
Apply Definitions & Materialize Features

Register your feature definitions with the Feast registry, then materialize feature values into the online store. Verify that materialization completed successfully.

In [ ]:
#Note : This should be always done in intialized feast repo. 
!feast apply

In [ ]:
#to materalize we need to know the start and end timestamp of the dataset
print("Start date to materialize",min(data['event_timestamp']))
print("End date to materialize",max(data['event_timestamp']))

In [ ]:
#to materalize i.e, to upload to the dataset to the online storage
!feast materialize "2025-09-17 10:40:17.102131+00:00" "2025-10-01 10:40:17.102131+00:00"

## Task 4
Fetch Features for Training (Offline Store)

Using Feast's offline retrieval, fetch historical features for your full training set. Train your IRIS classification model using the returned dataset. Your training code should pull features from Feast rather than reading the CSV directly — this is the key behavioral change.

In [ ]:
import feast
import joblib 
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo")

entity_df = data[["iris_id","event_timestamp"]]

print("Entity_Info")
print("------------------------\n\n")
entity_df.info()
print("------------------------\n\n")


training_df = store.get_historical_features(
    entity_df = entity_df,
    features = [
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
).to_df()

print("Retrived feature Info")
print("--------------------------\n\n")
training_df.info()
print("---------------------------\n\n")

X = training_df[
    [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
    ]
]

y = data["species"]

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X,y)

joblib.dump(model,"artifacts/model.joblib")

print("Model Trained Successfully")

## Task 5
Fetch Features for Inference (Online Store)

Simulate a real-time inference scenario: given one or more iris sample IDs, retrieve their features from the online store and pass them to your trained model for prediction. Demonstrate that the predictions are consistent with what you would get using the raw data.

In [ ]:
import joblib
from feast import FeatureStore
import pandas as pd

data = pd.read_csv("feature_repo/data/iris_data_adapted_for_feast.csv")
data["event_timestamp"] = pd.to_datetime(data["event_timestamp"])

model = joblib.load("artifacts/model.joblib")


store = FeatureStore(repo_path = "feature_repo")

print("\n\n")
iris_id = int(input("Enter the IRIS ID : "))

online_features = store.get_online_features(
    features = [
        "iris_features:sepal_length",
         "iris_features:sepal_width",
         "iris_features:petal_length",
         "iris_features:petal_width",
    ],
    entity_rows = [
        {"iris_id":iris_id},
    ],
).to_df()

X =  online_features[
    [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width"
    ]
]
if online_features.isnull().any().any():
    print(f"No features found for iris_id {iris_id}")
    exit()

predictions = model.predict(X)

print("\n\n")
print("The predicted species are")
print("------------------------------------")
print(predictions[0])

#comparing predictions with the raw data
raw = (data[data["iris_id"]==iris_id].sort_values("event_timestamp").iloc[-1])
raw_features = raw.drop(labels=["iris_id", "species", "event_timestamp", "created_timestamp"])

X = pd.DataFrame([raw_features])

raw_prediction = model.predict(X)

print("\n\n")
print("The actual raw dataset prediction")
print("------------------------------------------")
print( raw_prediction[0])

## Task 6 Optional
BigQuery Backend

Replace the local SQLite backend with BigQuery for either the offline store, the online store, or both. Document any configuration changes and trade-offs you observe.

In [ ]:
#To know the whether project is present or not.
gcloud config get-value project

#command to create dataset named feast
!bq mk feast

In [ ]:
#command to list the big query dataset.
!bq ls

In [ ]:
#Load the dataset table
!bq load \
--source_format=PARQUET \
feast.iris_data \
~/IITMBS_22F1001871_MLOPS_WEEKLY_ASSIGNMENT/feature_repo/data/iris_data_adapted_for_feast.parquet

In [ ]:
#verify table
!bq ls feast

In [ ]:
#After making changes to the yaml and feature definitons file
#Don't forget to delete the registry.db and online_db files available in data folder
!feast apply

In [ ]:
#Again
!feast materialize "2025-09-17 10:40:17.102131+00:00" "2025-10-01 10:40:17.102131+00:00"